In [28]:
import numpy as np

In [29]:
# 1. CONSTANTS & SETUP
A = 200000 / np.log(101)
B = 0.07
N_PLAYERS = 5400  # Total participants
BUDGET = 100
start_min = 0
start_max = 100

In [30]:
# 2. PRE-COMPUTE OPTIMAL DETERMINISTIC PAYOFFS
    # We find the best integer x for every possible budget S from 0 to 100
v_lookup = np.zeros((BUDGET + 1, 3))
for s in range(BUDGET + 1):
    best_prod = 0
    for x in range(s + 1):
        y = s - x
        # f(x) * g(y)
        prod = (A * np.log(1 + x)) * (B * y)
        if prod > best_prod:
            best_prod = prod
            v_lookup[s] = (x, y, best_prod)

def run_simulation(max_iterations = 50, suppress_print=False):
    # 3. INITIALIZE PLAYERS
    # Starting with a random distribution of z (investment in Asset 3)
    player_z = np.random.randint(start_min, start_max, N_PLAYERS)

    # 4. ITERATIVE BEST RESPONSE
    for iteration in range(max_iterations):
        changed = False
        # Shuffle player order to avoid ordering bias
        indices = np.random.permutation(N_PLAYERS)
        
        for i in indices:
            old_z = player_z[i]
            others_z = np.delete(player_z, i)
            sorted_others = np.sort(others_z)
            
            best_payoff = -BUDGET # The worst you can do is waste your entire budget with no return
            best_z = old_z
            
            # Test every possible integer z for the current player
            for test_z in range(BUDGET + 1):
                # COMPETITION RANKING: 1 + (count of players strictly better than you)
                # searchsorted(side='right') finds the index where test_z would be 
                # inserted after any existing entries of the same value.
                num_strictly_better = len(sorted_others) - np.searchsorted(sorted_others, test_z, side='right')
                rank = 1 + num_strictly_better
                
                # Multiplier h(z)
                h_z = 0.9 - ((rank - 1) / (N_PLAYERS - 1)) * 0.8
                
                # Total Payoff = [Deterministic Product * Multiplier] - Total Investment
                # Since we assume full budget utilization (or at least S+z):
                x, y, v_opt = v_lookup[BUDGET - test_z]
                current_payoff = (v_opt * h_z) - (x+y+test_z)
                
                if current_payoff > best_payoff:
                    best_payoff = current_payoff
                    best_z = test_z
            
            if best_z != old_z:
                player_z[i] = best_z
                changed = True
        
        # If no player wants to move, we've reached a Nash Equilibrium
        if not changed and not suppress_print:
            print(f"Nash Equilibrium reached after {iteration + 1} iterations.")
            break
    else:
        print(f"Reached max iterations ({max_iterations}) without full convergence.")

    # 5. RESULTS
    unique_vals, counts = np.unique(player_z, return_counts=True)
    if not suppress_print:
        print("\n--- Final Investment Distribution for Asset 3 (z) ---")
        for val, count in zip(unique_vals, counts):
            print(f"z = {val}: {count} players")
    
    return player_z

In [31]:
'''
dist = []
for _ in range(1):  # Run multiple simulations to observe variability
    z_values = run_simulation(suppress_print=True)
    if len(set(z_values)) == 1:
        dist.append(z_values[0])
print(len(dist), "out of 100 simulations converged to a single z value.")
'''

'\ndist = []\nfor _ in range(1):  # Run multiple simulations to observe variability\n    z_values = run_simulation(suppress_print=True)\n    if len(set(z_values)) == 1:\n        dist.append(z_values[0])\nprint(len(dist), "out of 100 simulations converged to a single z value.")\n'

In [32]:
run_simulation()

Nash Equilibrium reached after 2 iterations.

--- Final Investment Distribution for Asset 3 (z) ---
z = 35: 5400 players


array([35, 35, 35, ..., 35, 35, 35], shape=(5400,))

In [43]:
def run_simulation_fixed(fixed_z_val=0, fixed_fraction=0.3, max_iterations=50):
    n_fixed = int(N_PLAYERS * fixed_fraction)
    n_free = N_PLAYERS - n_fixed

    fixed_z = np.full(n_fixed, fixed_z_val)
    free_z = np.random.randint(0, 41, n_free)

    for iteration in range(max_iterations):
        changed = False
        indices = np.random.permutation(n_free)

        for i in indices:
            old_z = free_z[i]
            others_z = np.concatenate([free_z[np.arange(n_free) != i], fixed_z])
            sorted_others = np.sort(others_z)

            best_payoff = -BUDGET
            best_z = old_z

            for test_z in range(BUDGET + 1):
                num_strictly_better = len(sorted_others) - np.searchsorted(sorted_others, test_z, side='right')
                rank = 1 + num_strictly_better
                h_z = 0.9 - ((rank - 1) / (N_PLAYERS - 1)) * 0.8
                x, y, v_opt = v_lookup[BUDGET - test_z]
                current_payoff = (v_opt * h_z) - (x + y + test_z)

                if current_payoff > best_payoff:
                    best_payoff = current_payoff
                    best_z = test_z

            if best_z != old_z:
                free_z[i] = best_z
                changed = True

        if not changed:
            print(f"Nash Equilibrium reached after {iteration + 1} iterations.")
            break
    else:
        print(f"Reached max iterations ({max_iterations}) without full convergence.")

    all_z = np.concatenate([free_z, fixed_z])
    unique_vals, counts = np.unique(all_z, return_counts=True)
    print(f"\n--- z distribution (30% fixed at z={fixed_z_val}) ---")
    for val, count in zip(unique_vals, counts):
        print(f"z = {val}: {count} players")

    return free_z

run_simulation_fixed(fixed_z_val=0, fixed_fraction=0.3)

Nash Equilibrium reached after 2 iterations.

--- z distribution (30% fixed at z=0) ---
z = 0: 1620 players
z = 27: 3780 players


array([27, 27, 27, ..., 27, 27, 27], shape=(3780,))

In [44]:
ITERATIONS = 100
LEARNING_RATE = 0.1

In [45]:
# 3. INITIALIZE PROBABILITY DISTRIBUTIONS
# Each player has a weight for each z in [0, 100]
# Using log-weights (logits) for numerical stability
logits = np.zeros((N_PLAYERS, BUDGET + 1)) 

def get_probs(player_logits):
    exp_logits = np.exp(player_logits - np.max(player_logits))
    return exp_logits / np.sum(exp_logits)

# 4. SIMULATION LOOP
for i in range(ITERATIONS):
    # Sample a choice for each player based on their current distribution
    current_choices = np.array([
        np.random.choice(BUDGET + 1, p=get_probs(logits[p])) 
        for p in range(N_PLAYERS)
    ])
    
    # Calculate payoffs for the current round
    round_payoffs = np.zeros(N_PLAYERS)
    for p in range(N_PLAYERS):
        z_p = current_choices[p]
        others_z = np.delete(current_choices, p)
        
        # Competition Ranking (1 + count strictly better)
        rank = 1 + np.sum(others_z > z_p)
        h_z = 0.9 - ((rank - 1) / (N_PLAYERS - 1)) * 0.8
        x, y, prod = v_lookup[BUDGET - z_p]
        round_payoffs[p] = (prod * h_z) - (x+y+z_p)

    # Update Logits (Reinforcement Learning / Multiplicative Weights)
    # Players "reward" the z-value they chose based on the payoff received
    for p in range(N_PLAYERS):
        chosen_z = current_choices[p]
        logits[p, chosen_z] += LEARNING_RATE * round_payoffs[p]

    if i % 10 == 0:
        avg_z = np.mean(current_choices)
        print(f"Iteration {i}: Avg z across field = {avg_z:.2f}")

# 5. ANALYSIS
final_probs = get_probs(np.mean(logits, axis=0))
print("\n--- Final Equilibrium Strategy ---")
for z, p in enumerate(final_probs):
    if p > 0.01: # Only print significant strategies
        print(f"Invest z={z}: {p*100:.1f}% probability")

Iteration 0: Avg z across field = 51.15
Iteration 10: Avg z across field = 50.02
Iteration 20: Avg z across field = 50.02
Iteration 30: Avg z across field = 50.02
Iteration 40: Avg z across field = 50.02
Iteration 50: Avg z across field = 50.02
Iteration 60: Avg z across field = 50.02
Iteration 70: Avg z across field = 50.02
Iteration 80: Avg z across field = 50.02
Iteration 90: Avg z across field = 50.02

--- Final Equilibrium Strategy ---
Invest z=35: 100.0% probability
